# Axes, Grid, and Camera

**Part I · Visualization** — Tutorial 09

Control scene framing and reference geometry. You will learn to:

- Use `Axis`, `Grid`, `Axes2D`, and `Axes3D` as explicit scene objects.
- Style them (`AxisStyle`, `GridStyle`, `Axes2DStyle`, `Axes3DStyle`).
- Configure the camera: auto-fit, explicit `CameraConfig3d`, and the
  `View2DConfig` / `View3dConfig` input specs.
- Update the camera at runtime with `set_camera()`.


## Setup


In [1]:
from pytanga.geometry import Direction, Plane, Point, Sphere
from pytanga.viz import (
    Axes2D, Axes2DStyle, Axes3D, Axis, AxisStyle, CameraConfig3d, Grid, GridStyle,
    LabelStyle, View2DConfig, View3dConfig, Visualizer, get_camera_view2d,
    get_camera_view3d,
)


## 1. `Axis`, `Grid`, `Axes2D`, and `Axes3D`

Grids and axes are explicit scene objects (not hard-coded helpers). `Axis` is a
single axis from `start` to `end`; `Grid` is a coordinate grid spanned by
`dir_u`/`dir_v`; `Axes2D`/`Axes3D` draw the coordinate axes (two directions /
three directions).


In [2]:
viz = Visualizer(title="Axes & grid", add_default_axes=False, add_default_grid=False)

# A single X axis with value labels every 2 units
viz.add(Axis(start=(0, 0, 0), end=(10, 0, 0), major_interval=2.0, label="X"))

# A grid in the XY plane
viz.add(Grid(dir_u=(1, 0, 0), dir_v=(0, 1, 0), range_u=(-5, 5), range_v=(-3, 3)))

# Axes3D draws three directions at once
viz.add(Axes3D(range_u=(-5, 5), range_v=(-5, 5), range_w=(0, 5), labels=("X", "Y", "Z")))

viz.add(Point(3, 2, 0), color="#ffcc00", label="P")
viz.display_snapshot()


## 2. Axis styles and value labels

`AxisStyle` controls the axis name label (`label_style`) and numeric value
labels (`value_style`) separately. Value labels are drawn at each major
interval; `value_format` is a Python format specifier.


In [3]:
viz = Visualizer(title="Axes — styles", add_default_axes=False, add_default_grid=False)

viz.add(
    Axis(start=(0, 0, 0), end=(0, 0, 4), major_interval=1.0, label="Z"),
    style=AxisStyle(
        color="#44aaff",
        value_style=LabelStyle(rotation=45, offset_2d=(0, 10)),
        label_style=LabelStyle(offset_2d=(0, 20)),
    ),
)
viz.add(
    Axes2D(origin=(0, 0), range_u=(-5, 5), range_v=(-5, 5), labels=("X", "Y")),
    style=Axes2DStyle(u=AxisStyle(color="#ff6666"), v=AxisStyle(color="#6666ff")),
)
viz.display_snapshot()


## 3. Default axes + grid behaviour

By default every scene automatically receives a default `Axes3D` (or `Axes2D`
for `space_dim=2`) and a `Grid`. Suppress either with the constructor flags
`add_default_axes` / `add_default_grid` (both `True` by default).


In [4]:
viz3 = Visualizer()                 # 3D → default XYZ axes + XZ grid
viz2 = Visualizer(space_dim=2)      # 2D → default XY axes + XY grid
viz_none = Visualizer(add_default_axes=False, add_default_grid=False)  # neither


## 4. Camera — auto-fit and explicit `CameraConfig3d`

With `camera=None` (default) the camera auto-fits the scene's bounding box.
`flush(fit_camera=True)` re-fits at runtime. `CameraConfig3d` sets an explicit
perspective camera (any field left `None` is auto-computed).


In [5]:
# Auto-fit (default), then re-fit at runtime:
viz = Visualizer(title="Camera — auto-fit", add_default_axes=False, add_default_grid=False)
viz.add(Point(2, 0, 0), color="#ff4444")
viz.add(Point(-2, 1, 0), color="#44ff44")
viz.flush(fit_camera=True)
viz.display_snapshot()

# Explicit perspective camera:
viz2 = Visualizer(
    title="Camera — explicit",
    add_default_axes=False,
    add_default_grid=False,
    camera=CameraConfig3d(position=(0, 15, 0), target=(0, 0, 0), fov=30),
)
viz2.add(Point(2, 0, 0), color="#ff4444", label="$P$")
viz2.display_snapshot()


## 5. `View2DConfig` / `View3dConfig` input specs

`View2DConfig` (an orthographic view rectangle) and `View3dConfig` (a virtual
plane) are pure **input** specs — pass them directly to `Visualizer(camera=...)`
or convert them with `get_camera_view2d()` / `get_camera_view3d()` /
`get_camera()`.


In [6]:
# 2D orthographic view (deduces space_dim=2 automatically):
viz = Visualizer(
    title="Camera — View2DConfig",
    add_default_axes=False,
    add_default_grid=False,
    camera=View2DConfig(xmin=-4.0, xmax=4.0, ymin=-3.0, ymax=3.0),
)
viz.add(Point(2, 1, 0), color="#ff4444", label="$P$")
viz.display_snapshot()

# 3D plane-based view:
cam3d = get_camera_view3d(View3dConfig(
    point=(0.0, 0.0, 0.0), normal=(0.5, 0.4, 1.0), extent_u=6.0, extent_v=5.0, fov=45.0,
))
viz3 = Visualizer(title="Camera — View3dConfig", add_default_axes=False, add_default_grid=False, camera=cam3d)
viz3.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.3)
viz3.display_snapshot()


## 6. Runtime camera updates — `set_camera()`

`set_camera()` updates a scene's camera without restarting the viewer. It
accepts a `CameraConfig`, `View2DConfig`, or `View3dConfig` (which is converted
via `get_camera()`).


In [7]:
viz = Visualizer(title="Camera — set_camera", add_default_axes=False, add_default_grid=False)
viz.add(Point(2, 1, 0), color="#ff4444", label="$P$")

viz.set_camera(View2DConfig(xmin=0, xmax=8, ymin=0, ymax=6))
# viz.set_camera(CameraConfig3d(position=(0, 8, 0), target=(0, 0, 0)))
viz.flush()
viz.display_snapshot()
print("camera updated")


camera updated


## 7. Orbit controls

- **3D** (`space_dim=3`): left-drag rotates, middle/Shift+left pans, scroll or
  right-drag zooms.
- **2D** (`space_dim=2`): left/right-drag pans, scroll zooms; orbit rotation is
  locked. A 2D scene without an explicit camera defaults to a top-down
  orthographic camera, so `flush(fit_camera=True)` recenters it correctly.


## 8. Explicit ticks, line positions, and `Plane` spans

`Axis` exposes explicit `ticks=` (`(value, label)` pairs); `Grid` exposes
`line_positions_u` / `line_positions_v`. The `Plane` renderer honors
`span_u` / `span_v` (edge `Direction` vectors that define a parallelogram) and
the scalar `extent`; `position` / `normal` / `up` accept `Point()` /
`Direction()` objects.


In [8]:
viz = Visualizer(title="Axes — ticks & plane", add_default_axes=False, add_default_grid=False)

viz.add(Axis(start=(0, 0, 0), end=(6, 0, 0), ticks=[(0, "0"), (3, "mid"), (6, "end")]))
viz.add(Grid(line_positions_u=[-2, -1, 0, 1, 2], line_positions_v=[-1, 0, 1], range_u=(-2, 2), range_v=(-1, 1)))

# Plane with explicit span vectors (edge vectors defining a parallelogram):
viz.add(Plane(point=Point(0, 0, 3), normal=Direction(0, 0, 1), span_u=Direction(4, 0, 0), span_v=Direction(0, 2, 0)), opacity=0.25)

viz.display_snapshot()


## Visual Examples

A 2D framed scene and a 3D framed scene, exported via `export_snapshot()`.


In [9]:
# 2D: explicit axes + grid + camera rectangle
viz2 = Visualizer(
    title="Axes & camera — 2D",
    camera=View2DConfig(xmin=-5, xmax=5, ymin=-4, ymax=4),
    add_default_axes=False,
    add_default_grid=False,
)
viz2.add(Axes2D((0, 0), range_u=(-5, 5), range_v=(-4, 4), labels=("X", "Y")))
viz2.add(Grid((0, 0), range_u=(-5, 5), range_v=(-4, 4), interval_u=1, interval_v=1))
viz2.add(Point(3, 2, 0), color="#ff4444", label="$P$")
viz2.display_snapshot()

# 3D: explicit perspective camera + axes + grid
viz3 = Visualizer(
    title="Axes & camera — 3D",
    camera=CameraConfig3d(position=(10, 6, 12), target=(0, 0, 0), fov=45),
    add_default_axes=False,
    add_default_grid=False,
)
viz3.add(Axes3D(range_u=(-5, 5), range_v=(-5, 5), range_w=(0, 5), labels=("X", "Y", "Z")))
viz3.add(Grid(range_u=(-5, 5), range_v=(-5, 5)))
viz3.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.3)
viz3.display_snapshot()


## Summary

| Task | API |
|---|---|
| Single axis | `Axis(start, end, major_interval, label=...)` |
| Grid | `Grid(dir_u, dir_v, range_u, range_v)` |
| Axes groups | `Axes2D(...)` / `Axes3D(...)` |
| Axis styles | `AxisStyle(color, label_style, value_style)` |
| Default axes/grid | `Visualizer(add_default_axes=, add_default_grid=)` |
| Auto-fit | `flush(fit_camera=True)` |
| Explicit 3D camera | `CameraConfig3d(position, target, fov)` |
| 2D view spec | `View2DConfig(xmin, xmax, ymin, ymax)` |
| 3D view spec | `View3dConfig(point, normal, extent_u, extent_v)` |
| Builders | `get_camera_view2d` / `get_camera_view3d` / `get_camera` |
| Runtime update | `set_camera(...)` |

**Next:** [10 — Coordinate System](../10_coordinate_system/).
